### Household Cluster Analysis and Profiling

#### Business Goal
The goal of this analysis is to identify and describe distinct household shopping-behavior segments so decision-makers have a clear view of how customers differ. The analysis is descriptive and does not prescribe how the business should target, prioritize, or otherwise use the resulting segments.

#### Notebook Goal
This notebook profiles the clustering results using the original-scale shopping features and available household demographics. Demographics are used only as supplemental context and were not included as clustering features.

#### Import Libraries
Import the libraries used for data handling and cluster analysis.

In [1]:
# Core analysis libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Improve plot styling
sns.set()

# Show all columns during EDA
pd.set_option('display.max_columns', None)

# Display all dataframes generated within a cell
from IPython.display import display

In [2]:
# File path locations
from pathlib import Path

# File paths
PROJECT_ROOT = Path.cwd().resolve().parent
CLEAN_DATA_DIR = PROJECT_ROOT / "data/clean_data"
RAW_DATA_DIR = PROJECT_ROOT / "data/raw_data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

#### Import Cluster-Labeled Data
Load the household-level data with the retained labels from each clustering model so the resulting segments can be analyzed in their original shopping units.

In [3]:
# Import data and show first five rows
hh_shopping_labeled = pd.read_parquet(CLEAN_DATA_DIR / "04_modeling_with_labels.parquet")
hh_shopping_labeled.head()

,household_key,num_trips,recency_days,total_spend,avg_basket_value,total_units_purchased,avg_days_between_trips,avg_basket_size,log10_total_spend,log10_avg_basket_value,log10_avg_basket_size,kmeans_label,agg_label,gmm_label,dbscan_label
0,1,86,5,4330.16,50.35,1997,7.7,23.2,3.636504,1.701999,1.365488,0,2,0,0
1,2,45,43,1954.34,43.43,834,12.8,18.5,3.291000,1.637790,1.267172,0,2,4,0
2,3,47,8,2653.21,56.45,8540,12.8,181.7,3.423772,1.751664,2.259355,0,0,3,0
3,4,30,84,1200.11,40.00,382,18.0,12.7,3.079221,1.602060,1.103804,0,2,1,0
4,5,40,8,779.06,19.48,245,15.8,6.1,2.891571,1.289589,0.785330,5,2,4,0


In [4]:
# Number of rows, columns
hh_shopping_labeled.shape

(2500, 15)

In [5]:
# Data types and nulls
hh_shopping_labeled_types_null_counts = pd.DataFrame()
hh_shopping_labeled_types_null_counts["Data Type"] = hh_shopping_labeled.dtypes
hh_shopping_labeled_types_null_counts["Count of Nulls"] = hh_shopping_labeled.isna().sum()
hh_shopping_labeled_types_null_counts

,Data Type,Count of Nulls
household_key,int64,0
num_trips,int64,0
recency_days,int64,0
total_spend,float64,0
avg_basket_value,float64,0
total_units_purchased,int64,0
avg_days_between_trips,float64,0
avg_basket_size,float64,0
log10_total_spend,float64,0
log10_avg_basket_value,float64,0


#### Import Demographic Information
Demographic information is available for 801 of the 2,500 households and will be used only to add context to the behavior-based segments.

In [6]:
demo_data = pd.read_csv(RAW_DATA_DIR / "hh_demographic.csv")
demo_data.head()

,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,household_key
0,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,1
1,45-54,A,50-74K,Homeowner,2 Adults No Kids,2,None/Unknown,7
2,25-34,U,25-34K,Unknown,2 Adults Kids,3,1,8
3,25-34,U,75-99K,Homeowner,2 Adults Kids,4,2,13
4,45-54,B,50-74K,Homeowner,Single Female,1,None/Unknown,16


In [7]:
demo_data.shape

(801, 8)

In [8]:
demo_data["household_key"].nunique()

801

In [9]:
# Data types and nulls
demo_types_null_counts = pd.DataFrame()
demo_types_null_counts["Data Type"] = demo_data.dtypes
demo_types_null_counts["Count of Nulls"] = demo_data.isna().sum()
demo_types_null_counts

,Data Type,Count of Nulls
AGE_DESC,object,0
MARITAL_STATUS_CODE,object,0
INCOME_DESC,object,0
HOMEOWNER_DESC,object,0
HH_COMP_DESC,object,0
HOUSEHOLD_SIZE_DESC,object,0
KID_CATEGORY_DESC,object,0
household_key,int64,0


In [10]:
# Standardize column names
demo_data.columns = (
    demo_data.columns.str.strip()
             .str.lower()
             .str.replace(" ", "_")
             .str.replace("-", "_")
)

demo_data.columns

Index(['age_desc', 'marital_status_code', 'income_desc', 'homeowner_desc',
       'hh_comp_desc', 'household_size_desc', 'kid_category_desc',
       'household_key'],
      dtype='object')

#### Review Demographic Information
The demographic data includes age, marital-status code, income, homeownership, household composition, household size, and children. Some categories include unknown values, so these fields are used only as descriptive context.

In [11]:
# List percentages for each unique value in each column
for c in demo_data.drop("household_key", axis=1).columns:
    print(demo_data[c].value_counts(ascending=False, dropna=False, normalize=True).round(2))

age_desc
45-54    0.36
35-44    0.24
25-34    0.18
65+      0.09
55-64    0.07
19-24    0.06
Name: proportion, dtype: float64
marital_status_code
U    0.43
A    0.42
B    0.15
Name: proportion, dtype: float64
income_desc
50-74K       0.24
35-49K       0.21
75-99K       0.12
25-34K       0.10
15-24K       0.09
Under 15K    0.08
125-149K     0.05
100-124K     0.04
150-174K     0.04
250K+        0.01
175-199K     0.01
200-249K     0.01
Name: proportion, dtype: float64
homeowner_desc
Homeowner          0.63
Unknown            0.29
Renter             0.05
Probable Renter    0.01
Probable Owner     0.01
Name: proportion, dtype: float64
hh_comp_desc
2 Adults No Kids    0.32
2 Adults Kids       0.23
Single Female       0.18
Single Male         0.12
Unknown             0.09
1 Adult Kids        0.06
Name: proportion, dtype: float64
household_size_desc
2     0.40
1     0.32
3     0.14
5+    0.08
4     0.07
Name: proportion, dtype: float64
kid_category_desc
None/Unknown    0.70
1               0.1

#### Merge Cluster and Demographic Data
Demographic information is left-joined to the labeled household data so all 2,500 households remain available for behavioral analysis. Households without demographic records remain in the cluster profiles but cannot contribute to demographic descriptions.

In [12]:
# Perform left join
clusters_with_demo = hh_shopping_labeled.merge(
    demo_data,
    on="household_key",
    how="left"
)

clusters_with_demo.head()

,household_key,num_trips,recency_days,total_spend,avg_basket_value,total_units_purchased,avg_days_between_trips,avg_basket_size,log10_total_spend,log10_avg_basket_value,log10_avg_basket_size,kmeans_label,agg_label,gmm_label,dbscan_label,age_desc,marital_status_code,income_desc,homeowner_desc,hh_comp_desc,household_size_desc,kid_category_desc
0,1,86,5,4330.16,50.35,1997,7.7,23.2,3.636504,1.701999,1.365488,0,2,0,0,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown
1,2,45,43,1954.34,43.43,834,12.8,18.5,3.291000,1.637790,1.267172,0,2,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,47,8,2653.21,56.45,8540,12.8,181.7,3.423772,1.751664,2.259355,0,0,3,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,30,84,1200.11,40.00,382,18.0,12.7,3.079221,1.602060,1.103804,0,2,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,40,8,779.06,19.48,245,15.8,6.1,2.891571,1.289589,0.785330,5,2,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Check all column names
clusters_with_demo.columns

Index(['household_key', 'num_trips', 'recency_days', 'total_spend',
       'avg_basket_value', 'total_units_purchased', 'avg_days_between_trips',
       'avg_basket_size', 'log10_total_spend', 'log10_avg_basket_value',
       'log10_avg_basket_size', 'kmeans_label', 'agg_label', 'gmm_label',
       'dbscan_label', 'age_desc', 'marital_status_code', 'income_desc',
       'homeowner_desc', 'hh_comp_desc', 'household_size_desc',
       'kid_category_desc'],
      dtype='object')

In [14]:
# Data types and nulls
clusters_with_demo_types_null_counts = pd.DataFrame()
clusters_with_demo_types_null_counts["Data Type"] = clusters_with_demo.dtypes
clusters_with_demo_types_null_counts["Count of Nulls"] = clusters_with_demo.isna().sum()
clusters_with_demo_types_null_counts

,Data Type,Count of Nulls
household_key,int64,0
num_trips,int64,0
recency_days,int64,0
total_spend,float64,0
avg_basket_value,float64,0
total_units_purchased,int64,0
avg_days_between_trips,float64,0
avg_basket_size,float64,0
log10_total_spend,float64,0
log10_avg_basket_value,float64,0


### K-Means Cluster Profiles
K-Means is profiled using the five behavioral features that created the clusters, shown here in their original units rather than their transformed or scaled values.

**Cluster 0:** Higher-value-per-trip, moderate-frequency shoppers.  
**Cluster 1:** Infrequent shoppers who have not purchased recently.  
**Cluster 2:** Extremely inactive/unusual households.  
**Cluster 3:** Large-basket, fairly frequent and active shoppers.  
**Cluster 4:** Extremely frequent and currently active shoppers.  
**Cluster 5:** Lower-value, moderate-frequency shoppers.

In [15]:
# Shopping analysis
kmeans_shopping_analysis = clusters_with_demo.groupby("kmeans_label").agg(
    cluster_size=("household_key", "count"),
    avg_num_trips=("num_trips", "mean"),
    avg_recency_days=("recency_days", "mean"),
    avg_days_between_trips=("avg_days_between_trips", "mean"),
    avg_basket_size=("avg_basket_size", "mean"),
    avg_basket_value=("avg_basket_value", "mean")
).round(2)

kmeans_shopping_analysis

,cluster_size,avg_num_trips,avg_recency_days,avg_days_between_trips,avg_basket_size,avg_basket_value
kmeans_label,,,,,,
0,621,63.04,19.37,21.40,33.62,44.66
1,73,21.73,297.60,55.72,748.70,30.28
2,4,1.25,476.25,680.50,8.50,23.75
3,1017,108.81,14.53,10.31,1482.83,34.99
4,178,418.26,2.66,1.75,1111.70,22.80
5,607,83.42,21.46,14.78,58.11,15.44


#### K-Means Cluster 0 Analysis
Cluster 0 contains higher-value-per-trip, moderate-frequency shoppers and has the highest average basket value of any segment. Among households with demographic information, two-adult households without children are the most common household type, 70% are homeowners, and the largest income group is $50–74K at 22%.

In [16]:
kmeans_zero = clusters_with_demo[clusters_with_demo["kmeans_label"] == 0]
kmeans_zero.shape

(621, 22)

In [17]:
# Demographic columns
demo_cols = [
    'age_desc',
    'marital_status_code',
    'income_desc',
    'homeowner_desc', 
    'hh_comp_desc', 
    'household_size_desc',
    'kid_category_desc'
]

In [18]:
# Determine number of records with available demographic information
kmeans_zero[demo_cols].notna().sum()

age_desc               166
marital_status_code    166
income_desc            166
homeowner_desc         166
hh_comp_desc           166
household_size_desc    166
kid_category_desc      166
dtype: int64

In [19]:
# Check largest percentage of known demographic information
for d in demo_cols:
    print(kmeans_zero[d].value_counts(ascending=False, dropna=True, normalize=True).round(2).head(1))

age_desc
45-54    0.38
Name: proportion, dtype: float64
marital_status_code
A    0.46
Name: proportion, dtype: float64
income_desc
50-74K    0.22
Name: proportion, dtype: float64
homeowner_desc
Homeowner    0.7
Name: proportion, dtype: float64
hh_comp_desc
2 Adults No Kids    0.34
Name: proportion, dtype: float64
household_size_desc
2    0.45
Name: proportion, dtype: float64
kid_category_desc
None/Unknown    0.69
Name: proportion, dtype: float64


#### K-Means Cluster 1 Analysis
Cluster 1 contains infrequent shoppers who have not purchased recently, with long gaps between shopping trips. No demographic information is available for this segment, so its profile is based entirely on shopping behavior.

In [20]:
kmeans_one = clusters_with_demo[clusters_with_demo["kmeans_label"] == 1]
kmeans_one.shape

(73, 22)

In [21]:
# Determine number of records with available demographic information
kmeans_one[demo_cols].notna().sum()

age_desc               0
marital_status_code    0
income_desc            0
homeowner_desc         0
hh_comp_desc           0
household_size_desc    0
kid_category_desc      0
dtype: int64

#### K-Means Cluster 2 Analysis
Cluster 2 contains only four households and represents the most extreme inactivity in the K-Means solution. Its very small size and lack of demographic information mean this segment should be interpreted cautiously.

#### K-Means Cluster 3 Analysis
Cluster 3 contains large-basket, fairly frequent, and active shoppers. Among households with demographic information, two-adult households without children are the most common household type, 65% are homeowners, and the largest income group is $50–74K.

In [22]:
kmeans_three = clusters_with_demo[clusters_with_demo["kmeans_label"] == 3]
kmeans_three.shape

(1017, 22)

In [23]:
# Determine number of records with available demographic information
kmeans_three[demo_cols].notna().sum()

age_desc               417
marital_status_code    417
income_desc            417
homeowner_desc         417
hh_comp_desc           417
household_size_desc    417
kid_category_desc      417
dtype: int64

In [24]:
# Check largest percentage of known demographic information
for d in demo_cols:
    print(kmeans_three[d].value_counts(ascending=False, dropna=True, normalize=True).round(2).head(1))

age_desc
45-54    0.37
Name: proportion, dtype: float64
marital_status_code
A    0.46
Name: proportion, dtype: float64
income_desc
50-74K    0.27
Name: proportion, dtype: float64
homeowner_desc
Homeowner    0.65
Name: proportion, dtype: float64
hh_comp_desc
2 Adults No Kids    0.35
Name: proportion, dtype: float64
household_size_desc
2    0.42
Name: proportion, dtype: float64
kid_category_desc
None/Unknown    0.69
Name: proportion, dtype: float64


#### K-Means Cluster 4 Analysis
Cluster 4 is the most active segment, with very high shopping frequency, very recent activity, and short gaps between trips. Among households with demographic information, 62% are ages 35–54, 74% live in one- or two-person households, and 55% are homeowners.

Marital status is unknown for 47% of the demographic subset, so no strong conclusion can be made from that field. The children field is also recorded as None/Unknown for 72% of households.

In [25]:
kmeans_four = clusters_with_demo[clusters_with_demo["kmeans_label"] == 4]
kmeans_four.shape

(178, 22)

In [26]:
# Determine number of records with available demographic information
kmeans_four[demo_cols].notna().sum()

age_desc               116
marital_status_code    116
income_desc            116
homeowner_desc         116
hh_comp_desc           116
household_size_desc    116
kid_category_desc      116
dtype: int64

In [27]:
# Check largest percentage of known demographic information
for d in demo_cols:
    print(kmeans_four[d].value_counts(ascending=False, dropna=True, normalize=True).round(2).head(2))

age_desc
45-54    0.33
35-44    0.29
Name: proportion, dtype: float64
marital_status_code
U    0.47
A    0.38
Name: proportion, dtype: float64
income_desc
35-49K    0.21
50-74K    0.21
Name: proportion, dtype: float64
homeowner_desc
Homeowner    0.55
Unknown      0.32
Name: proportion, dtype: float64
hh_comp_desc
2 Adults No Kids    0.26
2 Adults Kids       0.23
Name: proportion, dtype: float64
household_size_desc
1    0.40
2    0.34
Name: proportion, dtype: float64
kid_category_desc
None/Unknown    0.72
1               0.16
Name: proportion, dtype: float64


#### K-Means Cluster 5 Analysis
Cluster 5 contains lower-value, moderate-frequency shoppers. Among households with demographic information, 50% are ages 35–54, the two largest income groups are $35–49K and $50–74K, 51% are homeowners, and 72% live in one- or two-person households.

Marital status is unknown for 55% of the demographic subset, so it does not provide a reliable description of this segment. The children field is also recorded as None/Unknown for 71% of households.

In [28]:
kmeans_five = clusters_with_demo[clusters_with_demo["kmeans_label"] == 5]
kmeans_five.shape

(607, 22)

In [29]:
# Determine number of records with available demographic information
kmeans_five[demo_cols].notna().sum()

age_desc               102
marital_status_code    102
income_desc            102
homeowner_desc         102
hh_comp_desc           102
household_size_desc    102
kid_category_desc      102
dtype: int64

In [30]:
# Check largest percentage of known demographic information
for d in demo_cols:
    print(kmeans_five[d].value_counts(ascending=False, dropna=True, normalize=True).round(2).head(2))

age_desc
45-54    0.31
35-44    0.19
Name: proportion, dtype: float64
marital_status_code
U    0.55
A    0.27
Name: proportion, dtype: float64
income_desc
35-49K    0.26
50-74K    0.20
Name: proportion, dtype: float64
homeowner_desc
Homeowner    0.51
Unknown      0.40
Name: proportion, dtype: float64
hh_comp_desc
2 Adults Kids       0.25
2 Adults No Kids    0.24
Name: proportion, dtype: float64
household_size_desc
1    0.43
2    0.29
Name: proportion, dtype: float64
kid_category_desc
None/Unknown    0.71
1               0.16
Name: proportion, dtype: float64


### Analysis of Agglomerative Clustering
Agglomerative clustering produces several recognizable behavioral groups, but its cluster structure is more uneven and its overall separation metrics were weaker than K-Means. It is retained here as a comparison rather than profiled in detail.

In [31]:
# Shopping analysis
agg_shopping_analysis = clusters_with_demo.groupby("agg_label").agg(
    cluster_size=("household_key", "count"),
    avg_num_trips=("num_trips", "mean"),
    avg_recency_days=("recency_days", "mean"),
    avg_days_between_trips=("avg_days_between_trips", "mean"),
    avg_basket_size=("avg_basket_size", "mean"),
    avg_basket_value=("avg_basket_value", "mean")
).round(2)

agg_shopping_analysis

,cluster_size,avg_num_trips,avg_recency_days,avg_days_between_trips,avg_basket_size,avg_basket_value
agg_label,,,,,,
0,1385,132.70,11.50,9.36,1232.06,32.23
1,8,2.00,313.12,480.12,8.09,23.83
2,654,62.73,17.06,21.58,19.05,40.64
3,80,41.01,275.84,33.15,988.63,29.39
4,40,654.98,2.52,1.11,408.47,12.20
5,333,66.58,36.58,18.73,7.22,14.42


### Analysis of GMM Clusters
GMM produces more balanced cluster sizes, but the groups showed substantially greater overlap in the model evaluation. Its behavioral breakdown is retained for comparison, but the weaker separation does not provide a strong reason to carry it forward for detailed profiling.

In [32]:
# Shopping analysis
gmm_shopping_analysis = clusters_with_demo.groupby("gmm_label").agg(
    cluster_size=("household_key", "count"),
    avg_num_trips=("num_trips", "mean"),
    avg_recency_days=("recency_days", "mean"),
    avg_days_between_trips=("avg_days_between_trips", "mean"),
    avg_basket_size=("avg_basket_size", "mean"),
    avg_basket_value=("avg_basket_value", "mean")
).round(2)

gmm_shopping_analysis

,cluster_size,avg_num_trips,avg_recency_days,avg_days_between_trips,avg_basket_size,avg_basket_value
gmm_label,,,,,,
0,886,108.59,4.91,6.48,768.03,33.50
1,97,67.32,90.95,13.19,11.60,25.36
2,210,10.31,129.00,79.66,630.01,32.16
3,433,59.50,35.33,14.16,1079.66,32.14
4,441,38.01,16.59,20.14,14.66,32.74
5,433,298.03,2.45,2.52,1221.47,27.26


### Analysis of DBSCAN Clusters
DBSCAN produced strong silhouette and Davies-Bouldin scores, but almost the entire population was assigned to one cluster. This makes it useful for highlighting unusual observations, but not as a meaningful multi-segment household solution.

In [33]:
# Shopping analysis
dbscan_shopping_analysis = clusters_with_demo.groupby("dbscan_label").agg(
    cluster_size=("household_key", "count"),
    avg_num_trips=("num_trips", "mean"),
    avg_recency_days=("recency_days", "mean"),
    avg_days_between_trips=("avg_days_between_trips", "mean"),
    avg_basket_size=("avg_basket_size", "mean"),
    avg_basket_value=("avg_basket_value", "mean")
).round(2)

dbscan_shopping_analysis

,cluster_size,avg_num_trips,avg_recency_days,avg_days_between_trips,avg_basket_size,avg_basket_value
dbscan_label,,,,,,
-1,49,247.65,235.67,117.27,1007.88,29.56
0,2441,108.26,19.82,13.83,722.12,31.64
1,3,8.33,271.33,51.00,1523.77,48.86
2,7,7.14,455.57,28.97,14.17,30.77


### Final Model Assessment

K-Means remains the most defensible segmentation from the models tested because it produces multiple distinct and interpretable shopping-behavior groups while also performing well across the shared clustering metrics. Profiling the clusters in their original units confirms meaningful differences in shopping frequency, recency, basket size, and basket value.

These segments are analytical descriptions rather than recommendations for business action. Decisions about whether and how the groups are used belong to business stakeholders and depend on objectives and constraints outside the scope of this analysis.

### Limitations and Future Research

Demographic information is available for only 801 of the 2,500 households, and coverage varies considerably across clusters. Several demographic fields also contain Unknown or None/Unknown categories, so demographic findings should be treated as supplemental descriptions rather than complete profiles.

The segmentation intentionally uses shopping behavior only. A future analysis could include selected demographic variables as clustering features and compare the resulting segmentation with the current behavior-only approach.